# Step 03 — Grounded VLM Semantic Oracle

This step creates a **model-derived semantic target** for every advertisement:

\[
S=[s_H,s_P,s_C,s_O]
\]

The VLM is deliberately **not asked to predict gaze**. It judges the ad's intended semantic and persuasive structure, so the later Human↔VLM alignment test is not circular.

### The six layers

1. **Prompt ensemble:** semantic architecture, persuasive funnel, designed hierarchy.
2. **Self-consistency:** repeated runs; disagreement is measured with Jensen–Shannon divergence.
3. **Grounding stability:** repeated H/P/C boxes are compared using IoU.
4. **Semantic Hierarchy Drift:** compare each edited condition with original A.
5. **Intended semantic route:** e.g. `H>P>C`, saved for later scanpath comparison.
6. **Counterfactual ablation (optional):** blur one grounded region and measure what semantic function is lost.

Primary output for Step 04:

`outputs/vlm_semantic_master.csv`

In [ ]:
from pathlib import Path
import hashlib, itertools, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from scipy.spatial.distance import jensenshannon
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

print("Working directory:", Path.cwd().resolve())

## 1. Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# Optional robustness run later:
# MODEL_ID = "Qwen/Qwen3-VL-30B-A3B-Instruct"
# USE_4BIT = True

USE_4BIT = False
RUNS_PER_PROMPT = 3
TEMPERATURE = 0.35
TOP_P = 0.85
MAX_NEW_TOKENS = 650
SEED_BASE = 20261001

IMAGE_ROOT = None
# Example:
# IMAGE_ROOT = Path("/home/walidtaib/Desktop/ads")

CONDITIONS = ["A","B","C","D","E"]
VALID_EXTENSIONS = {".jpg",".jpeg",".png",".webp"}

RUN_COUNTERFACTUAL_ABLATION = False
COUNTERFACTUAL_MAX_IMAGES = 10

OUTPUT_DIR = Path.cwd() / "outputs"
CACHE_DIR = Path.cwd() / "cache"
OVERLAY_DIR = OUTPUT_DIR / "grounding_overlays"
CF_DIR = OUTPUT_DIR / "counterfactual_examples"

for d in [OUTPUT_DIR, CACHE_DIR, OVERLAY_DIR, CF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PROMPT_ENSEMBLE = json.loads((Path.cwd()/"prompt_ensemble.json").read_text())

print("Model:", MODEL_ID)
print("Prompts:", list(PROMPT_ENSEMBLE))
print("Runs per prompt:", RUNS_PER_PROMPT)

## 2. GPU diagnostics

In [ ]:
import torch, transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory/1024**3:.2f} GiB")

## 3. Discover A–E advertisement images

In [ ]:
def parse_condition_image(path):
    path = Path(path)
    stem = path.stem

    parent_cond = None
    for parent in [path.parent, *path.parents]:
        if parent.name.upper() in CONDITIONS:
            parent_cond = parent.name.upper()
            break

    m = re.search(r"(\d{1,3}).*?([A-E])$", stem, re.I)
    if m:
        return m.group(2).upper(), int(m.group(1))

    n = re.search(r"(\d{1,3})", stem)
    if n and parent_cond:
        return parent_cond, int(n.group(1))

    return None, None

def recognized_images(root):
    rows = []
    root = Path(root)
    if not root.exists():
        return rows
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS:
            c, i = parse_condition_image(p)
            if c in CONDITIONS and i is not None:
                rows.append((p,c,i))
    return rows

def autodetect_root():
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.append(base)
        try:
            candidates.extend([p for p in base.iterdir() if p.is_dir()])
        except Exception:
            pass

    scored, seen = [], set()
    for c in candidates:
        try:
            c = c.resolve()
        except Exception:
            continue
        if c in seen:
            continue
        seen.add(c)
        n = len(recognized_images(c))
        if n:
            scored.append((n,c))

    scored.sort(reverse=True, key=lambda x:x[0])
    if scored:
        print("Top candidate roots:")
        for n,p in scored[:8]:
            print(f"{n:4d} recognized images -> {p}")
        return scored[0][1]
    return None

if IMAGE_ROOT is None:
    IMAGE_ROOT = autodetect_root()

if IMAGE_ROOT is None:
    raise RuntimeError("No ad-image root detected. Set IMAGE_ROOT manually.")

IMAGE_ROOT = Path(IMAGE_ROOT).resolve()

rows = []
for p,c,i in recognized_images(IMAGE_ROOT):
    rows.append({
        "condition":c,
        "image_id":i,
        "image_path":str(p.resolve()),
        "filename":p.name
    })

inventory = pd.DataFrame(rows).sort_values(["image_id","condition","filename"])

dups = inventory.duplicated(["condition","image_id"], keep=False)
if dups.any():
    print("WARNING: duplicate image candidates:")
    display(inventory[dups])

inventory = inventory.drop_duplicates(["condition","image_id"],keep="first").reset_index(drop=True)
display(inventory)
inventory.to_csv(OUTPUT_DIR/"vlm_image_inventory.csv",index=False)

## 4. Load Qwen3-VL

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

kwargs = {
    "device_map":"auto",
    "trust_remote_code":True,
    "torch_dtype":torch.bfloat16,
}

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
    kwargs.pop("torch_dtype",None)

model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, **kwargs)
model.eval()

print("Loaded:", MODEL_ID)
if hasattr(model,"hf_device_map"):
    print(model.hf_device_map)

## 5. Parsing, validation, and inference helpers

In [ ]:
def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?","",text,flags=re.I).strip()
    text = re.sub(r"```$","",text).strip()
    try:
        return json.loads(text)
    except Exception:
        a,b = text.find("{"), text.rfind("}")
        if a < 0 or b <= a:
            raise ValueError("No JSON object found.")
        return json.loads(text[a:b+1])

def normalize_budget(d):
    arr = np.array([d.get(k,0) for k in ["H","P","C","O"]],dtype=float)
    arr = np.maximum(arr,0)
    if not np.isfinite(arr).all() or arr.sum()<=0:
        raise ValueError(f"Invalid budget: {d}")
    arr /= arr.sum()
    return dict(zip(["H","P","C","O"],map(float,arr)))

def clean_bbox(x):
    if x is None or not isinstance(x,(list,tuple)) or len(x)!=4:
        return None
    try:
        b = [float(v) for v in x]
    except Exception:
        return None
    b = [min(1000,max(0,v)) for v in b]
    if b[2]<=b[0] or b[3]<=b[1]:
        return None
    return b

def sanitize(obj):
    budget = normalize_budget(obj["budget"])
    elements = {}

    for e in ["H","P","C"]:
        raw = (obj.get("elements",{}) or {}).get(e,{}) or {}
        present = bool(raw.get("present",False))
        box = clean_bbox(raw.get("bbox")) if present else None
        try:
            conf = float(np.clip(float(raw.get("confidence",np.nan)),0,1))
        except Exception:
            conf = np.nan

        elements[e] = {
            "present":present,
            "bbox":box,
            "confidence":conf,
            "text_or_description":str(raw.get("text_or_description",""))[:500]
        }

    order = obj.get("intended_order",[])
    if not isinstance(order,list):
        order=[]
    order = [str(x).upper() for x in order if str(x).upper() in ["H","P","C"]]

    def score(name):
        try:
            return float(np.clip(float(obj.get(name,np.nan)),0,100))
        except Exception:
            return np.nan

    return {
        "budget":budget,
        "elements":elements,
        "intended_order":order,
        "message_clarity":score("message_clarity"),
        "product_identity":score("product_identity"),
        "action_clarity":score("action_clarity"),
        "rationale_short":str(obj.get("rationale_short",""))[:1000]
    }

def input_device():
    return next(model.parameters()).device

@torch.inference_mode()
def infer_json(image,prompt,seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    messages=[{
        "role":"user",
        "content":[
            {"type":"image"},
            {"type":"text","text":prompt}
        ]
    }]

    rendered = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[rendered],
        images=[image],
        return_tensors="pt",
        padding=True
    )

    dev = input_device()
    inputs = {
        k:(v.to(dev) if hasattr(v,"to") else v)
        for k,v in inputs.items()
    }

    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P
    )

    n = inputs["input_ids"].shape[1]
    text = processor.batch_decode(
        out[:,n:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return sanitize(extract_json(text)), text

def cache_file(condition,image_id,prompt_name,run_idx):
    key = f"{MODEL_ID}|{condition}|{image_id}|{prompt_name}|{run_idx}|{TEMPERATURE}|{TOP_P}"
    digest = hashlib.sha256(key.encode()).hexdigest()[:16]
    return CACHE_DIR/f"{condition}_{image_id:02d}_{prompt_name}_{run_idx}_{digest}.json"

## 6. Run the prompt ensemble

With 100 images, 3 prompts and 3 runs:

\[
100\times3\times3 = 900
\]

VLM judgments. Every response is cached, so an interrupted run can continue.

In [ ]:
raw_rows=[]

for item in tqdm(inventory.to_dict("records"),desc="Ads"):
    condition=item["condition"]
    image_id=int(item["image_id"])
    image_path=Path(item["image_path"])
    image=Image.open(image_path).convert("RGB")

    for pidx,(pname,prompt) in enumerate(PROMPT_ENSEMBLE.items()):
        for run_idx in range(RUNS_PER_PROMPT):
            seed=(SEED_BASE+image_id*1000+CONDITIONS.index(condition)*100+pidx*10+run_idx)
            cp=cache_file(condition,image_id,pname,run_idx)

            if cp.exists():
                payload=json.loads(cp.read_text())
            else:
                payload={
                    "condition":condition,
                    "image_id":image_id,
                    "prompt_name":pname,
                    "run_idx":run_idx,
                    "seed":seed,
                    "status":"OK",
                    "error":"",
                    "clean":None,
                    "raw_text":""
                }
                try:
                    clean,text=infer_json(image,prompt,seed)
                    payload["clean"]=clean
                    payload["raw_text"]=text
                except Exception as e:
                    payload["status"]="ERROR"
                    payload["error"]=f"{type(e).__name__}: {e}"
                cp.write_text(json.dumps(payload,indent=2,ensure_ascii=False))

            row={
                "model_id":MODEL_ID,
                "condition":condition,
                "image_id":image_id,
                "image_path":str(image_path),
                "prompt_name":pname,
                "run_idx":run_idx,
                "seed":seed,
                "status":payload["status"],
                "error":payload.get("error","")
            }

            clean=payload.get("clean")
            if row["status"]=="OK" and clean is not None:
                for k in ["H","P","C","O"]:
                    row[f"s_{k}"]=clean["budget"][k]
                for e in ["H","P","C"]:
                    elem=clean["elements"][e]
                    row[f"{e}_present"]=elem["present"]
                    row[f"{e}_bbox"]=json.dumps(elem["bbox"])
                    row[f"{e}_grounding_confidence"]=elem["confidence"]
                    row[f"{e}_description"]=elem["text_or_description"]

                row["intended_order"]=">".join(clean["intended_order"])
                row["message_clarity"]=clean["message_clarity"]
                row["product_identity"]=clean["product_identity"]
                row["action_clarity"]=clean["action_clarity"]
                row["rationale_short"]=clean["rationale_short"]

            raw_rows.append(row)

raw=pd.DataFrame(raw_rows)
display(raw.groupby("status").size().reset_index(name="n"))
raw.to_csv(OUTPUT_DIR/"vlm_semantic_scores_all_runs.csv",index=False)
raw[raw["status"]!="OK"].to_csv(OUTPUT_DIR/"vlm_semantic_errors.csv",index=False)

## 7. Consensus semantic target + uncertainty

In [ ]:
S_COLS=["s_H","s_P","s_C","s_O"]
valid=raw[raw["status"]=="OK"].copy()
if valid.empty:
    raise RuntimeError("No valid model outputs. Inspect vlm_semantic_errors.csv.")

def jsd_bits(p,q,eps=1e-12):
    p=np.clip(np.asarray(p,float),eps,None); p/=p.sum()
    q=np.clip(np.asarray(q,float),eps,None); q/=q.sum()
    return float(jensenshannon(p,q,base=2.0)**2)

def entropy_bits(p,eps=1e-12):
    p=np.clip(np.asarray(p,float),eps,None); p/=p.sum()
    return float(-(p*np.log2(p)).sum())

rows=[]
for (condition,image_id),sub in valid.groupby(["condition","image_id"]):
    arr=sub[S_COLS].to_numpy(float)
    s=arr.mean(axis=0); s/=s.sum()

    jsds=[
        jsd_bits(arr[i],arr[j])
        for i,j in itertools.combinations(range(len(arr)),2)
    ]
    uncertainty=float(np.mean(jsds)) if jsds else np.nan
    sd=arr.std(axis=0,ddof=1) if len(arr)>1 else np.zeros(4)

    orders=sub["intended_order"].dropna()
    modal=orders.mode().iloc[0] if len(orders) and len(orders.mode()) else ""
    agreement=float((orders==modal).mean()) if len(orders) else np.nan
    sorted_s=np.sort(s)[::-1]

    rows.append({
        "condition":condition,
        "image_id":int(image_id),
        "s_H":s[0],"s_P":s[1],"s_C":s[2],"s_O":s[3],
        "sd_H":sd[0],"sd_P":sd[1],"sd_C":sd[2],"sd_O":sd[3],
        "semantic_entropy_bits":entropy_bits(s),
        "semantic_entropy_normalized":entropy_bits(s)/2.0,
        "hierarchy_margin":float(sorted_s[0]-sorted_s[1]),
        "mean_pairwise_prompt_JSD":uncertainty,
        "semantic_reliability":1.0-uncertainty if np.isfinite(uncertainty) else np.nan,
        "modal_intended_order":modal,
        "order_agreement":agreement,
        "message_clarity_mean":sub["message_clarity"].mean(),
        "product_identity_mean":sub["product_identity"].mean(),
        "action_clarity_mean":sub["action_clarity"].mean(),
        "n_successful_judgments":len(sub)
    })

consensus=pd.DataFrame(rows).sort_values(["image_id","condition"])
display(consensus.head(30))
consensus.to_csv(OUTPUT_DIR/"vlm_semantic_consensus.csv",index=False)

## 8. Grounding stability with repeated-box IoU

In [ ]:
def parse_box(x):
    if x is None or (isinstance(x,float) and np.isnan(x)):
        return None
    try:
        return clean_bbox(json.loads(x)) if isinstance(x,str) else clean_bbox(x)
    except Exception:
        return None

def iou(a,b):
    if a is None or b is None:
        return np.nan
    ax1,ay1,ax2,ay2=a
    bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1)
    ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1)
    aa=max(0,ax2-ax1)*max(0,ay2-ay1)
    ab=max(0,bx2-bx1)*max(0,by2-by1)
    union=aa+ab-inter
    return inter/union if union>0 else np.nan

grows=[]
for (condition,image_id),sub in valid.groupby(["condition","image_id"]):
    row={"condition":condition,"image_id":int(image_id)}

    for e in ["H","P","C"]:
        boxes=[parse_box(x) for x in sub[f"{e}_bbox"]]
        boxes=[b for b in boxes if b is not None]
        vals=[iou(a,b) for a,b in itertools.combinations(boxes,2)]
        vals=[v for v in vals if np.isfinite(v)]

        row[f"{e}_n_boxes"]=len(boxes)
        row[f"{e}_mean_pairwise_iou"]=float(np.mean(vals)) if vals else np.nan
        row[f"{e}_consensus_bbox"]=(
            json.dumps(np.median(np.asarray(boxes),axis=0).tolist())
            if boxes else None
        )

    stable=[
        row[f"{e}_mean_pairwise_iou"]
        for e in ["H","P","C"]
        if np.isfinite(row[f"{e}_mean_pairwise_iou"])
    ]
    row["grounding_stability"]=float(np.mean(stable)) if stable else np.nan
    grows.append(row)

grounding=pd.DataFrame(grows)
consensus=consensus.merge(grounding,on=["condition","image_id"],how="left")

display(grounding.head(30))
grounding.to_csv(OUTPUT_DIR/"vlm_grounding_stability.csv",index=False)
consensus.to_csv(OUTPUT_DIR/"vlm_semantic_consensus_with_grounding.csv",index=False)

## 9. Draw consensus grounding overlays

In [ ]:
def denorm(box,w,h):
    return [
        int(round(box[0]/1000*w)),
        int(round(box[1]/1000*h)),
        int(round(box[2]/1000*w)),
        int(round(box[3]/1000*h))
    ]

lookup=inventory.set_index(["condition","image_id"])

for _,r in tqdm(consensus.iterrows(),total=len(consensus),desc="Overlays"):
    key=(r["condition"],int(r["image_id"]))
    if key not in lookup.index:
        continue

    p=lookup.loc[key,"image_path"]
    if isinstance(p,pd.Series):
        p=p.iloc[0]

    img=Image.open(p).convert("RGB")
    draw=ImageDraw.Draw(img)
    w,h=img.size

    for e in ["H","P","C"]:
        b=parse_box(r.get(f"{e}_consensus_bbox"))
        if b is None:
            continue
        px=denorm(b,w,h)
        draw.rectangle(px,width=max(2,int(min(w,h)*0.004)))
        draw.text((px[0]+3,max(0,px[1]-14)),e)

    img.save(OVERLAY_DIR/f"{int(r['image_id']):02d}{r['condition']}_grounding.png")

print("Saved overlays:",OVERLAY_DIR)

## 10. Semantic Hierarchy Drift relative to original A

\[
SHD_{i,X}=JSD(S_{i,A},S_{i,X})
\]

\[
SemanticPreservation=1-SHD.
\]

In [ ]:
drows=[]
for image_id,sub in consensus.groupby("image_id"):
    base=sub[sub["condition"]=="A"]
    if base.empty:
        continue

    sA=base.iloc[0][S_COLS].to_numpy(float)

    for _,r in sub.iterrows():
        sx=r[S_COLS].to_numpy(float)
        d=jsd_bits(sA,sx)
        drows.append({
            "image_id":int(image_id),
            "condition":r["condition"],
            "semantic_hierarchy_drift_JSD":d,
            "semantic_preservation":1.0-d
        })

drift=pd.DataFrame(drows)
consensus=consensus.merge(drift,on=["image_id","condition"],how="left")

drift.to_csv(OUTPUT_DIR/"semantic_hierarchy_drift_vs_A.csv",index=False)
consensus.to_csv(OUTPUT_DIR/"vlm_semantic_master.csv",index=False)
display(drift.head(30))

## 11. Main visualizations

In [ ]:
# 1) Mean semantic composition
cm=(
    consensus.groupby("condition",observed=True)[S_COLS]
    .mean()
    .reindex(CONDITIONS)
)

fig,ax=plt.subplots(figsize=(11,6))
left=np.zeros(len(cm))
y=np.arange(len(cm))

for c in S_COLS:
    vals=cm[c].to_numpy()
    ax.barh(y,vals,left=left,label=c.replace("s_",""))
    left+=vals

ax.set_yticks(y); ax.set_yticklabels(cm.index)
ax.set_xlim(0,1)
ax.set_xlabel("Mean VLM semantic budget")
ax.set_ylabel("Condition")
ax.set_title("VLM Semantic Composition Across Conditions")
ax.legend(title="Component",bbox_to_anchor=(1.02,0.5),loc="center left")
ax.grid(axis="x",alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"vlm_semantic_composition_by_condition.png",dpi=300,bbox_inches="tight")
fig.savefig(OUTPUT_DIR/"vlm_semantic_composition_by_condition.pdf",bbox_inches="tight")
plt.show()

# 2) Semantic hierarchy drift heatmap
heat=(
    drift.pivot(index="image_id",columns="condition",values="semantic_hierarchy_drift_JSD")
    .reindex(columns=CONDITIONS)
    .sort_index()
)

fig,ax=plt.subplots(figsize=(8,10))
im=ax.imshow(heat.to_numpy(),aspect="auto")
ax.set_xticks(np.arange(len(heat.columns))); ax.set_xticklabels(heat.columns)
ax.set_yticks(np.arange(len(heat.index))); ax.set_yticklabels(heat.index)
ax.set_xlabel("Condition"); ax.set_ylabel("Base advertisement")
ax.set_title("Semantic Hierarchy Drift Relative to Original A")

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v=heat.iloc[i,j]
        if pd.notna(v):
            ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=7)

fig.colorbar(im,ax=ax,label="JSD from A")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"semantic_hierarchy_drift_heatmap.png",dpi=300,bbox_inches="tight")
fig.savefig(OUTPUT_DIR/"semantic_hierarchy_drift_heatmap.pdf",bbox_inches="tight")
plt.show()

# 3) Reliability vs grounding stability
fig,ax=plt.subplots(figsize=(9,7))
for cond in CONDITIONS:
    sub=consensus[consensus["condition"]==cond]
    ax.scatter(
        sub["semantic_reliability"],
        sub["grounding_stability"],
        s=35,alpha=0.65,label=cond
    )

ax.set_xlabel("Semantic reliability (1 - prompt JSD)")
ax.set_ylabel("Grounding stability (mean box IoU)")
ax.set_title("Semantic Reliability vs Spatial Grounding Stability")
ax.grid(alpha=0.2)
ax.legend(title="Condition")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"semantic_reliability_vs_grounding_stability.png",dpi=300,bbox_inches="tight")
fig.savefig(OUTPUT_DIR/"semantic_reliability_vs_grounding_stability.pdf",bbox_inches="tight")
plt.show()

## 12. Optional counterfactual semantic ablation

For region \(k\in\{H,P,C\}\), blur only that grounded region and re-score the advertisement.

Example:

\[
\Delta ActionClarity_C
=
ActionClarity_{original}
-
ActionClarity_{CTA\ masked}.
\]

The direct semantic score and the counterfactual drop are independent signals.

In [ ]:
CF_PROMPT = (
    "Evaluate this advertisement without predicting gaze. "
    "Return ONLY valid JSON with four 0..100 scores: "
    '{"message_clarity":0,"product_identity":0,"action_clarity":0,"semantic_completeness":0}'
)

def blur_region(img,box):
    out=img.copy().convert("RGB")
    if box is None:
        return out
    w,h=out.size
    x1,y1,x2,y2=denorm(box,w,h)
    crop=out.crop((x1,y1,x2,y2))
    radius=max(3,int(min(w,h)*0.035))
    crop=crop.filter(ImageFilter.GaussianBlur(radius=radius))
    out.paste(crop,(x1,y1))
    return out

def clean_cf(obj):
    result={}
    for k in ["message_clarity","product_identity","action_clarity","semantic_completeness"]:
        try:
            result[k]=float(np.clip(float(obj[k]),0,100))
        except Exception:
            result[k]=np.nan
    return result

@torch.inference_mode()
def infer_cf(img,seed):
    torch.manual_seed(seed)
    messages=[{
        "role":"user",
        "content":[
            {"type":"image"},
            {"type":"text","text":CF_PROMPT}
        ]
    }]

    rendered=processor.apply_chat_template(
        messages,tokenize=False,add_generation_prompt=True
    )
    inp=processor(
        text=[rendered],images=[img],
        return_tensors="pt",padding=True
    )
    dev=input_device()
    inp={k:(v.to(dev) if hasattr(v,"to") else v) for k,v in inp.items()}
    out=model.generate(**inp,max_new_tokens=160,do_sample=False)
    n=inp["input_ids"].shape[1]
    txt=processor.batch_decode(out[:,n:],skip_special_tokens=True)[0]
    return clean_cf(extract_json(txt))

cf_rows=[]

if RUN_COUNTERFACTUAL_ABLATION:
    targets=(
        consensus.sort_values(
            ["semantic_reliability","grounding_stability"],
            ascending=False
        )
        .head(COUNTERFACTUAL_MAX_IMAGES)
    )

    lookup=inventory.set_index(["condition","image_id"])

    for _,r in tqdm(targets.iterrows(),total=len(targets),desc="Counterfactual"):
        key=(r["condition"],int(r["image_id"]))
        if key not in lookup.index:
            continue

        p=lookup.loc[key,"image_path"]
        if isinstance(p,pd.Series):
            p=p.iloc[0]

        img=Image.open(p).convert("RGB")
        seed=SEED_BASE+int(r["image_id"])*100
        base=infer_cf(img,seed)

        for j,e in enumerate(["H","P","C"]):
            box=parse_box(r.get(f"{e}_consensus_bbox"))
            if box is None:
                continue

            masked=blur_region(img,box)
            masked_scores=infer_cf(masked,seed+j+1)
            masked.save(CF_DIR/f"{int(r['image_id']):02d}{r['condition']}_mask_{e}.png")

            row={
                "condition":r["condition"],
                "image_id":int(r["image_id"]),
                "masked_element":e
            }

            for metric in ["message_clarity","product_identity","action_clarity","semantic_completeness"]:
                row[f"original_{metric}"]=base[metric]
                row[f"masked_{metric}"]=masked_scores[metric]
                row[f"drop_{metric}"]=base[metric]-masked_scores[metric]

            cf_rows.append(row)

    counterfactual=pd.DataFrame(cf_rows)
    display(counterfactual)
    counterfactual.to_csv(OUTPUT_DIR/"counterfactual_semantic_ablation.csv",index=False)
else:
    print("Counterfactual ablation is OFF. First inspect the main VLM results.")

## 13. Intended semantic route for Step 05

In [ ]:
route_table=consensus[
    [
        "condition","image_id",
        "modal_intended_order","order_agreement",
        "semantic_reliability","grounding_stability"
    ]
].copy()

display(route_table.head(30))
route_table.to_csv(OUTPUT_DIR/"vlm_intended_semantic_routes.csv",index=False)

route_counts=(
    route_table.groupby(["condition","modal_intended_order"],observed=True)
    .size()
    .reset_index(name="n_ads")
    .sort_values(["condition","n_ads"],ascending=[True,False])
)

display(route_counts)
route_counts.to_csv(OUTPUT_DIR/"vlm_intended_route_counts.csv",index=False)

## 14. Final audit / manual-review queue

In [ ]:
audit=consensus[
    [
        "condition","image_id",
        "s_H","s_P","s_C","s_O",
        "semantic_entropy_normalized",
        "hierarchy_margin",
        "semantic_reliability",
        "grounding_stability",
        "modal_intended_order",
        "order_agreement",
        "semantic_hierarchy_drift_JSD",
        "semantic_preservation",
        "n_successful_judgments"
    ]
].copy()

audit["low_semantic_reliability"]=audit["semantic_reliability"]<0.90
audit["low_grounding_stability"]=(
    audit["grounding_stability"].notna()
    & (audit["grounding_stability"]<0.30)
)
audit["needs_manual_review"]=(
    audit["low_semantic_reliability"]
    | audit["low_grounding_stability"]
)

display(
    audit.sort_values(
        ["needs_manual_review","semantic_reliability"],
        ascending=[False,True]
    ).head(50)
)

audit.to_csv(OUTPUT_DIR/"vlm_semantic_audit.csv",index=False)

## 15. Output summary

In [ ]:
print("STEP 03 COMPLETE")
print("="*75)
print("Core Step 04 file: outputs/vlm_semantic_master.csv")
print()
print("Important outputs:")
for name in [
    "vlm_semantic_scores_all_runs.csv",
    "vlm_semantic_consensus.csv",
    "vlm_grounding_stability.csv",
    "semantic_hierarchy_drift_vs_A.csv",
    "vlm_intended_semantic_routes.csv",
    "vlm_semantic_audit.csv"
]:
    print(" -",name)

print()
print("S is a semantic target, NOT a gaze prediction.")
print("Next: 04_human_vlm_alignment")